# 02 — The tool loop: ReAct by hand, twice

**What you'll learn**

- Define tools as data: a `Tool` dataclass whose JSON-schema `params` is all the model ever sees
- Run any tool through `run_tool` — errors come back as results, never as exceptions
- Build ReAct twice: as a bare text protocol parsed with `parse_json_loose`, then on native `tool_calls`
- The canonical loop `shoplab.loop.run_agent`: three stop reasons, `max_steps`, result truncation
- Point the loop at the full ops desk — `standard_tools()`, risky flags, the `Ledger` — and triage a real ticket

*Time: ~25 min. Cost: ~$0.01. Cached reruns are free.*

## An agent is a while loop with a model deciding each pass

Strip away the vocabulary and every agent in production is the same object: a model called in a loop. Each pass, the model reads the conversation so far and does one of two things — requests a tool call, or gives a final answer. Your code runs the tool, appends the result to the conversation, and calls the model again. That is the entire architecture; everything else this course builds — tracing, evals, budgets, approval gates, context management — is instrumentation bolted onto this one loop.

The pattern has a paper: *ReAct: Synergizing Reasoning and Acting in Language Models* (Yao et al., 2022, [arXiv:2210.03629](https://arxiv.org/abs/2210.03629)) showed that interleaving reasoning traces with actions and their observations beats reasoning or acting alone. This chapter builds ReAct twice. First as a bare text protocol, the way the paper ran it — so you see there is no magic, just constrained text. Then on the native tool-calling API every serious provider now ships. Same loop both times; only the wire format moves.

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import litellm
from dotenv import load_dotenv              # pip install -e ".[obs]" if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead

TEMPERATURE = 0                             # the whole course runs at temperature 0
litellm.drop_params = True                  # ignore params a provider does not support
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=".litellm_cache")  # reruns are ~free

### Phoenix observability (optional)

The frozen cell finds or starts the local Phoenix server and traces every LiteLLM call this notebook makes — including every step of every agent run below, which is exactly what you will want to browse when chapter 03 makes tracing the topic. Optional: skip it and nothing else changes.

In [ ]:
# optional: Phoenix tracing (see notebook 03)
import obs

obs.enable_phoenix()

## Tools are data the model reads, plus a function it never touches

A tool is four fields: a name, a one-sentence description, a JSON schema for the arguments, and the Python function that actually runs. The model receives only the first three, rendered as text. It cannot execute anything — it can only emit a request naming a tool, and your loop decides what happens next. The model proposes; the loop disposes. That separation is why approval gates (chapter 08) and injection defenses (chapter 09) are possible at all.

The `risky` flag marks tools that move money or ship goods. Nothing enforces it in this chapter — it is a bit of data waiting for chapter 08's gates.

Build the dataclass, then the desk's first three tools: two lookups over the chapter-00 world data, plus `search_policy` — already a plain function, now getting the schema that makes it callable by a model.

In [ ]:
from dataclasses import dataclass
from typing import Callable

@dataclass
class Tool:
    name: str            # what the model calls it
    description: str     # one sentence the model reads
    params: dict         # JSON schema for the arguments
    fn: Callable         # the Python that actually runs
    risky: bool = False  # moves money or goods; gated in ch08

In [ ]:
from shoplab import world

orders = {o["order_id"]: o for o in world.load_orders()}
customers = {c["customer_id"]: c for c in world.load_customers()}

def get_order(order_id):
    return orders.get(order_id, {"error": f"no such order {order_id}"})

def get_customer(customer_id):
    return customers.get(customer_id, {"error": f"no such customer {customer_id}"})

print(get_order("ORD-7301")["total_usd"], get_customer("CUST-01")["tier"])

In [ ]:
def str_arg(name):       # JSON schema for one required string argument
    return {"type": "object", "properties": {name: {"type": "string"}},
            "required": [name]}

TOOLS = {t.name: t for t in [
    Tool("get_order", "Look up an order by id: items, totals, status, dates.",
         str_arg("order_id"), get_order),
    Tool("get_customer", "Look up a customer by id: tier, flags, history.",
         str_arg("customer_id"), get_customer),
    Tool("search_policy", "Keyword-search the 12 store policy documents.",
         str_arg("query"), world.search_policy),
]}
print(list(TOOLS))

## A failing tool call is a result, not an exception

The loop's contract with tools has one clause: whatever happens, `run_tool` returns a JSON string. Unknown tool, wrong argument name, an id that matches nothing, a bug three frames down — all of it comes back as `{"error": ...}`. The reason is architectural, not cosmetic. In an agent, the model is the component that decides what to do next, so failures must reach the model as text it can read. An exception that kills the loop informs nobody; an error string becomes an observation the model can react to — retry, rephrase, route around. MCP bakes the same convention into its spec: tool execution errors travel inside the tool result, flagged `isError: true`, never as protocol-level errors ([MCP tools spec](https://modelcontextprotocol.io/specification/2026-07-28/server/tools)).

Provoke it three ways.

In [ ]:
import json

def run_tool(tools, name, args):
    if name not in tools:
        return json.dumps({"error": f"unknown tool {name}"})
    try:
        return json.dumps(tools[name].fn(**args))
    except Exception as exc:
        return json.dumps({"error": f"{type(exc).__name__}: {exc}"[:200]})

print(run_tool(TOOLS, "get_order", {"order_id": "ORD-9999"}))
print(run_tool(TOOLS, "teleport", {"to": "warehouse"}))
print(run_tool(TOOLS, "get_order", {"oops": "ORD-7301"}))

> **What you should see:** three JSON strings, each carrying an `"error"` key — a missed order id, an unknown tool, a `TypeError` naming the unexpected argument. Nothing raises. The loop that will sit on top of this never has to care.

## ReAct round one: the protocol is just text

Before tool-calling APIs existed, ReAct ran on plain completions, and building it that way first is the fastest cure for magical thinking. The system prompt orders the model to answer in two lines — THOUGHT, free-text reasoning, then ACTION, one JSON object that either requests a tool or delivers a final answer — and lists the tools as plain text. That prompt *is* the protocol; there is nothing underneath it.

The loop around it: call the model, pull the ACTION out of the reply with `parse_json_loose` (chapter 01's fence-tolerant parser, imported from its canonical home), execute the tool, and feed the result back as the next user message, prefixed `OBSERVATION:`. The model never sees anything but text it was told to expect.

In [ ]:
tool_lines = "\n".join(
    f"- {t.name}({', '.join(t.params['properties'])}): {t.description}"
    for t in TOOLS.values())
TEXT_SYSTEM = f"""You answer ops-desk questions for Larkspur Outfitters.
Work in steps. Every reply must be exactly two lines:
THOUGHT: one sentence of reasoning.
ACTION: one JSON object -- {{"tool": "<name>", "args": {{...}}}} to call a tool,
or {{"final": "<your answer>"}} once you can answer.
Available tools:
{tool_lines}
After a tool action, the next user message is OBSERVATION: <result>.
Never invent tool results. One action per reply."""
print(TEXT_SYSTEM)

In [ ]:
from shoplab.llm import complete, parse_json_loose

def run_text_agent(task, tools, max_steps=6):
    messages = [{"role": "system", "content": TEXT_SYSTEM},
                {"role": "user", "content": task}]
    for step in range(1, max_steps + 1):
        reply = complete(messages).choices[0].message.content
        print(f"--- step {step} ---\n{reply.strip()}\n")
        messages.append({"role": "assistant", "content": reply})
        action = parse_json_loose(reply)
        if "final" in action:
            return action["final"]
        obs = run_tool(tools, action.get("tool", ""), action.get("args") or {})
        print(f"observation: {obs[:120]}\n")
        messages.append({"role": "user", "content": "OBSERVATION: " + obs})

print("FINAL:", run_text_agent(
    "What is the total of order ORD-7301, and is that customer a vip?", TOOLS))

> **What you should see:** strict THOUGHT/ACTION alternation. The agent calls `get_order`, then `get_customer`, then answers; the final line reports a total of 391.50 dollars and confirms the customer is vip. Wording and step phrasing drift between runs and providers — the shape does not.

## ReAct round two: the same loop on native tool calls

Tool calling as text works, which is exactly why providers absorbed it: models are now trained to emit tool calls as structured API fields, so nobody has to parse THOUGHT/ACTION out of prose in production. (The idea that models can learn tool use predates the APIs — *Toolformer: Language Models Can Teach Themselves to Use Tools* (Schick et al., 2023, [arXiv:2302.04761](https://arxiv.org/abs/2302.04761)) had models teach themselves API calls.) You pass `tools=` in the OpenAI function format — name, description, parameters schema, the exact fields of your `Tool` — and the reply carries `msg.tool_calls` instead of prose. Same loop; the parsing moved behind the API.

In [ ]:
def to_openai_tools(tools):
    return [{"type": "function",
             "function": {"name": t.name, "description": t.description,
                          "parameters": t.params}}
            for t in tools.values()]

msgs = [{"role": "user",
         "content": "What did order ORD-7301 cost, shipping included?"}]
msg = complete(msgs, tools=to_openai_tools(TOOLS)).choices[0].message
print("content:   ", repr(msg.content))
print("tool_calls:", [(tc.function.name, tc.function.arguments)
                      for tc in msg.tool_calls])

> **What you should see:** `content` is empty or a one-line narration, and `tool_calls` holds exactly one `get_order` call — whose `arguments` field is a JSON *string*, not a dict. You parse it before executing; a model can and eventually will put garbage there.

One model call is half a ReAct step. To finish it: append the assistant message — tool calls and all — back onto the conversation, execute each requested tool, append each result as a `role: "tool"` message tagged with the call's id, and ask the model again.

In [ ]:
msgs.append(msg.model_dump(exclude_none=True))
for tc in msg.tool_calls:
    msgs.append({"role": "tool", "tool_call_id": tc.id,
                 "content": run_tool(TOOLS, tc.function.name,
                                     json.loads(tc.function.arguments))})
print(complete(msgs, tools=to_openai_tools(TOOLS)).choices[0].message.content)

> **What you should see:** a prose answer built from the observation, quoting a total of 391.50 dollars. Request, execute, respond — that round trip is one ReAct step, and the loop is just that step repeated.

## The finished form: `shoplab.loop.run_agent`

Wrap the step body in a `for` loop, add exits, and you have the runner the rest of the course instruments, evaluates, and defends. It stops three ways, recorded as `stop_reason`: the model called the `finish` tool (`"finish"`), it answered in plain text (`"text"`), or it ran out of turns (`"max_steps"`). Two hooks pass through: `on_step` lets observers watch each turn (chapter 03 hangs tracing off it), and `before_tool` is a veto point this chapter leaves unused — chapters 08 and 09 attach approval gates and injection defenses there. `max_result_chars` truncates oversized tool results before they flood the context.

The next cell is not a transcript. The region between the sentinel comments lives byte-identical in `src/shoplab/loop.py` — the validator compares them on every build — and `run_agent` leans on the canonical package homes of what you just built: `run_tool` and `to_openai_tools` live in `shoplab.tools`, `complete` in `shoplab.llm`. `AgentResult` is the small record it returns.

In [ ]:
import shoplab.llm                        # run_agent calls shoplab.llm.complete
from shoplab.tools import run_tool, to_openai_tools   # canonical forms of yours

@dataclass
class AgentResult:
    answer: dict | str | None             # finish args, plain text, or None
    steps: int
    messages: list
    stop_reason: str                      # "finish" | "text" | "max_steps"

# >>> shoplab.loop.run_agent
def run_agent(task, tools, *, model=None, system=None, max_steps=8,
              on_step=None, before_tool=None, max_result_chars=2000):
    """Run the tool-calling loop: one model call per step, execute every tool
    call it makes, stop on the finish tool, a plain-text answer, or max_steps."""
    messages = [{"role": "system", "content": system}] if system else []
    messages.append({"role": "user", "content": task})
    for step in range(1, max_steps + 1):
        r = shoplab.llm.complete(messages, model=model, tools=to_openai_tools(tools))
        msg = r.choices[0].message
        # real litellm messages become plain dicts; test fakes pass through as-is
        messages.append(msg.model_dump(exclude_none=True)
                        if hasattr(msg, "model_dump") else msg)
        if on_step:
            on_step(step, msg)
        if msg.tool_calls:
            for tc in msg.tool_calls:
                name, ok = tc.function.name, True
                try:
                    args = json.loads(tc.function.arguments or "{}")
                except ValueError:
                    ok, args = False, {}
                    result = json.dumps({"error": "tool arguments were not valid JSON"})
                else:
                    if before_tool is not None and before_tool(name, args) is False:
                        ok, result = False, '{"error": "blocked by policy"}'
                    else:
                        result = run_tool(tools, name, args)
                messages.append({"role": "tool", "tool_call_id": tc.id,
                                 "content": result[:max_result_chars]})
                if ok and name == "finish":
                    return AgentResult(answer=args, steps=step,
                                       messages=messages, stop_reason="finish")
        elif msg.content:
            return AgentResult(answer=msg.content, steps=step,
                               messages=messages, stop_reason="text")
    return AgentResult(answer=None, steps=max_steps, messages=messages,
                       stop_reason="max_steps")
# <<< shoplab.loop.run_agent

Laid over *ReAct: Synergizing Reasoning and Acting in Language Models* (Yao et al., 2022, [arXiv:2210.03629](https://arxiv.org/abs/2210.03629)), the loop is the paper's diagram translated into message roles:

| Paper concept | Plain English | Where it lives in code |
|---|---|---|
| Thought | reasoning before acting | assistant message `content` |
| Act | a structured tool request | `msg.tool_calls` (native) / the ACTION JSON (text) |
| Obs | the environment answering | the `role: "tool"` message / the OBSERVATION turn |
| Loop until answer | repeat, then stop | `for step in ...` around one `complete` call, ended by `finish`, text, or `max_steps` |

## The full desk: nine tools and a ledger

`shoplab.tools.standard_tools()` is the ops desk of docs/WORLD.md: the three tools you built, plus inventory, a no-`eval` calculator, the two risky write tools, `escalate`, and `finish` — the terminator that ends a run with a structured decision. The risky pair records every side effect to a `Ledger` you pass in, because an agent's actions should leave evidence you can audit after the fact — chapters 08 and 09 are built on reading it.

In [ ]:
from shoplab.tools import Ledger, standard_tools

ledger = Ledger()
desk = standard_tools(ledger)
for t in desk.values():
    tag = "RISKY" if t.risky else ("terminator" if t.name == "finish" else "")
    print(f"{t.name:<19} {tag:<11} {t.description}")

> **What you should see:** nine tools; exactly two flagged RISKY; `finish` last, as the terminator.

Now the desk's real job: a return ticket, end to end. The system prompt below does real work — it names the job, demands lookups before decisions, caps the tool budget, and pins the `finish` contract. The budget sentence earns its place: an earlier draft without it looked everything up, then kept searching policies for a rule that might not exist and ran out of steps. Prompts steer loops, not just answers — though steer is the precise word, as the transcript is about to show. The task is the ticket's structured fields rendered into one line, customer prose included.

In [ ]:
SYSTEM = ("You are the operations desk agent for Larkspur Outfitters. "
          "Use the tools to look up the order, the customer, and the relevant "
          "policy before deciding. You have a hard budget of six tool calls, so "
          "look nothing up twice, and keep any commentary to one short sentence "
          "per step. When you are sure, call finish with decision "
          "(approve_refund|partial_refund|replacement|store_credit|deny|escalate), "
          "policy_id, and refund_usd (number or null). "
          "Decisions must follow shop policy, not sympathy.")

def render_ticket(t):
    return (f"Ticket {t['ticket_id']} from {t['customer_id']} about order "
            f"{t['order_id']}, sku {t['sku']}, qty {t['qty']}, "
            f"condition {t['item_condition']}, days since delivery "
            f"{t['days_since_delivery']}, photo evidence {t['evidence_photo']}, "
            f"requested action {t['requested_action']}. "
            f"Customer writes: {t['reason_text']}")

In [ ]:
from shoplab.world import load_tickets

ticket = next(t for t in load_tickets()["train"] if t["ticket_id"] == "TKT-2205")
print(render_ticket(ticket), "\n")

def show(step, msg):
    names = [tc.function.name for tc in msg.tool_calls or []]
    print(f"step {step}:", ", ".join(names) or (msg.content or "")[:60])

result = run_agent(render_ticket(ticket), desk, system=SYSTEM,
                   max_steps=10, on_step=show)
print("\nstop:", result.stop_reason, "after", result.steps, "steps")
print("answer:", result.answer)
print("gold:  ", ticket["gold"])
print("ledger:", ledger.entries)

> **What you should see:** lookups first — order, then customer — a few policy searches, `calc` for the fee arithmetic, then `finish`, usually five to ten steps in. The answer should match the gold label: decision `partial_refund`, policy `pol-restocking`, amount 170.99 — a member-tier customer returning opened boots inside the 30-day window pays the 10 percent restocking fee, and 0.90 * 189.99 rounds to 170.99. The ledger stays empty: deciding a refund and moving money are different acts, and this prompt only asks for the decision. Now count the tool calls against the prompt: a hard budget of six, and the frozen run above spends seven before `finish` — three policy searches and a gratuitous inventory check. Nothing counted them. A prompt is a request, not enforcement; the budget sentence made over-searching rarer, and that is all a sentence can do. Enforcement is code, and it arrives with chapter 08's real budgets. Eyeballing one ticket proves nothing, though — chapter 04 builds the harness that grades all 40.

## Two blunt guards: `max_steps` and result truncation

Every loop ships with two circuit breakers from day one. `max_steps` bounds the worst case — a looping agent burns tokens, dollars, and rate limits until something external stops it, so the loop carries its own stop. `max_result_chars` (default 2000) truncates what each tool call may put into the context; one verbose tool result would otherwise ride along in every remaining step of the run. Both are crude. Chapter 08 replaces them with real budgets; until then, crude beats absent. Watch the loop hit the ceiling on the same ticket, then watch an observation get clipped.

In [ ]:
capped = run_agent(render_ticket(ticket), standard_tools(), system=SYSTEM,
                   max_steps=2)
print("stop:", capped.stop_reason, "after", capped.steps, "steps")
print("answer:", capped.answer)

clipped = run_agent(render_ticket(ticket), standard_tools(), system=SYSTEM,
                    max_steps=2, max_result_chars=80)
first_obs = next(m for m in clipped.messages if m.get("role") == "tool")
print("\nclipped observation:", repr(first_obs["content"]))

> **What you should see:** `stop_reason` `"max_steps"` and `answer` `None` after two steps. The loop returns a typed non-answer instead of hanging or guessing — callers can branch on it, and chapter 04's scorer will count it as a miss, not a crash. The second run shows the other guard at work: its first observation is `get_order`'s JSON cut off mid-field at 80 characters. Every later step would have inherited that clipped string — which is the guard's point, and its price, since truncated evidence can cost accuracy. The default of 2000 characters is roomy on purpose.

## What breaks first: the phantom tool

The classic loop-killer is the model requesting a tool that does not exist. Under native tool calling the trained models have become reluctant to do it — asked for a tool missing from their list, they tend to refuse outright. The text protocol has no such training wheels, so ask the text agent to try a `get_invoice` tool and watch the failure path work: the unknown-name error comes back as an observation, the model reads it, switches to a tool that exists, and still answers. That recovery is the whole argument for error-as-result — the same bad call raising an exception would have ended the run at step one.

In [ ]:
final = run_text_agent(
    "A teammate swears there is a get_invoice tool. Try it once for "
    "ORD-7305 -- report the exact error if it fails -- then look up the "
    "order the normal way. Your final answer must state both the error "
    "and the order total in dollars.", TOOLS)
print("FINAL:", final)

> **What you should see:** a `get_invoice` attempt answered by `{"error": "unknown tool get_invoice"}`, then a `get_order` call, then a final answer reporting both the error and the real total of 186.00 dollars. The error string did its job: course-correcting context.

## Recap

| Concept | One-liner |
|---|---|
| `Tool` | name + description + JSON-schema params + fn + `risky`; the model only ever sees the first three. |
| `run_tool` | the loop's one door into every tool; always a JSON string, never an exception. |
| Error-as-result | failures become observations the model can read, react to, and recover from. |
| Text-protocol ReAct | THOUGHT + ACTION JSON over plain completions — tool calling with zero API support. |
| Native tool calling | same loop; the action parsing moves into `tools=` and `msg.tool_calls`. |
| `run_agent` | the canonical loop in `shoplab.loop`; stops on `finish`, plain text, or `max_steps`. |
| `standard_tools()` | the nine-tool ops desk; the risky pair writes evidence to a `Ledger`. |
| `max_steps` / `max_result_chars` | the two blunt guards every loop needs before chapter 08's real budgets. |

## Exercises

1. Add a `get_product` tool backed by `world.load_products()` (return name, price, and the hazmat flag) to `TOOLS`, then rerun the phantom-tool cell asking for `get_product` instead of `get_invoice`. Which cells had to change for the outcome to flip from error to answer — and why is neither loop one of them?
2. `run_text_agent` trusts every ACTION to carry a `"tool"` or a `"final"` key. Investigate what happens when it carries neither — telling the agent its next ACTION must be exactly `{}` gets you there — then harden the loop: turn a keyless action into an `OBSERVATION` that names the problem, and confirm the model recovers.
3. Every call this chapter made left a row in `shoplab.llm.LEDGER`. Slice out the TKT-2205 run and print `prompt_tokens` per step. Explain the shape — the whole transcript is resent every step — and extrapolate what step 20 would cost. That curve is why chapter 10 exists.

**Next up:** chapter 03 cracks the loop open — hand-built spans and Phoenix traces that make every step, token, and dollar visible.